# 07 — V2 LLM features: a self-contained experiment

## 1. Objective

Notebook 03 found a specific weakness in the first LLM feature set. Seven of its ten features
were statistically associated with diagnosis, so the *constructs* were sound — but the
representation reached CV AUC 0.694, which is indistinguishable from simply counting the words
in the transcript (0.693). One feature assigned the same value to 92% of speakers, another was
largely a restatement of transcript length, and eight of the ten declared levels the model
never used.

The diagnosis: **a 3–5 level categorical label is too coarse a summary of a 60-word
transcript.** The model looks at the text, forms an impression, and throws almost all of it
away to emit one word.

V2 changes what we ask for, not what we ask about:

| | V1 | V2 |
|---|---|---|
| output | one ordinal level per construct | counts of observable events |
| evidence | none | verbatim spans, checkable against the transcript |
| length control | none | per-100-word rates computed by us, never by the model |
| prompt | described "two hidden categories" | label-blind and domain-blind |

This notebook runs that prompt over the 241 labelled training transcripts with Qwen 3.5 122B
and asks one question: **do the V2 features discriminate better than V1?**

All three outcomes are informative. V2 > V1 means the redesign worked. V2 ≈ V1 means the
problem is not the encoding and we should look at extraction reliability. V2 < V1 means Qwen
is producing inconsistent counts, which the groundedness check in section 6 will show.

In [1]:
!pip install llm-feature-gen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 1.7 MB/s eta 0:00:00


## 2. Run configuration

Everything that would change the results is recorded here and written to
`outputs/v2_run_metadata.json`. That file plus the cached raw responses is what makes this
reproducible — no separate machinery needed.

In [2]:
import json
import re
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

# ---- what to run -------------------------------------------------------------
RUN_LLM = True

MODEL           = "qwen3.5:122b"
BASE_URL        = "https://llm.vse.cz/ollama/v1/"
API_KEY         = "ollama"
PROMPT_VERSION  = "v2.0"

TEMPERATURE     = 0.0        # we want the same transcript to give the same counts
MAX_TOKENS      = 4096       # the library default of 2048 truncates a 13-field response
REQUEST_TIMEOUT = 900        # seconds; streaming keeps the connection alive but be generous
MAX_RETRIES     = 5
DISABLE_THINKING = True      # Qwen reasons before answering and can spend the whole budget on it

LIMIT_DOCS = None            # set to e.g. 20 for a quick trial run; None = all 241
SEED = 42

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
RAW_PATH  = OUT / "v2_raw_responses.jsonl"      # one JSON line per document, appended immediately
FEAT_PATH = OUT / "v2_features.csv"
META_PATH = OUT / "v2_run_metadata.json"

print(f"model {MODEL} | prompt {PROMPT_VERSION} | temperature {TEMPERATURE} | "
      f"max_tokens {MAX_TOKENS}")

model qwen3.5:122b | prompt v2.0 | temperature 0.0 | max_tokens 4096


## 3. Load the data

**241 labelled training transcripts only.** The 61-case test set is never read in this
notebook — the loader below simply does not look at `test/`.

The dataset locator handles the case that broke earlier runs: in a fresh Colab session there is
no `fileDataset/` folder, only the zip, and passing a string path that does not exist makes the
library raise a confusing "no non-empty text inputs" error.

In [3]:
def find_dataset(start: Path = Path(".")) -> Path:
    """Return a directory that directly contains overview/ and train/, unzipping if needed."""
    def ok(p: Path) -> bool:
        return (p / "train").is_dir() and (p / "overview").is_dir()

    for c in [start, start / "fileDataset", start / "data", start / "data" / "fileDataset",
              start / "fileDataset" / "fileDataset", start / "fileDataset_data", start.parent]:
        if c.is_dir() and ok(c):
            return c.resolve()

    zips = sorted(start.rglob("fileDataset.zip")) + sorted(start.parent.glob("fileDataset.zip"))
    if not zips:
        raise FileNotFoundError(
            "Could not find the dataset. Put fileDataset.zip (or an unpacked folder containing "
            "overview/ and train/) next to this notebook."
        )
    target = (start / "fileDataset_data").resolve()
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(target)
    print(f"unpacked {zips[0].name} -> {target}")
    for c in [target, target / "fileDataset"]:
        if ok(c):
            return c.resolve()
    raise FileNotFoundError(f"nothing usable inside {target}")


DATA = find_dataset()

rows = []
for sub, label in [("train/negative", 0), ("train/positive", 1)]:
    for path in sorted((DATA / sub).glob("*.txt")):
        rows.append({"file": path.name, "label": label,
                     "text": path.read_text(encoding="utf-8").strip()})
train = pd.DataFrame(rows)
train["n_word"] = train.text.str.split().str.len()

if LIMIT_DOCS:
    train = train.groupby("label", group_keys=False).apply(
        lambda g: g.head(max(1, int(LIMIT_DOCS * len(g) / len(rows))))).reset_index(drop=True)

print(f"dataset root: {DATA}")
print(f"{len(train)} training transcripts | {int(train.label.sum())} positive "
      f"({train.label.mean():.1%}) | median {train.n_word.median():.0f} words")
assert train.file.is_unique and len(train) > 0
print("test/ was not read")

FileNotFoundError: Could not find the dataset. Put fileDataset.zip (or an unpacked folder containing overview/ and train/) next to this notebook.

## 4. The V2 prompt

Two properties matter and both are asserted below.

**Label-blind:** nothing tells the model that categories exist. The library's default discovery
prompt describes "two hidden text categories", which is how phrases like *"one group tends
to… while the other…"* ended up in the V1 feature descriptions.

**Domain-blind:** the prompt also never says what the study is about. A prohibition such as
"do not infer cognitive impairment" would itself reveal the domain, so the instruction forbids
inference about the speaker in general terms instead.

Note the output semantics, which differ by field. `named_entities` and `regions_referenced` are
**sets** — distinct things, each listed once. Everything else is a list of **occurrences**, so
two hedges in one sentence are two entries. Section 7 turns both into counts.

In [ ]:
V2_PROMPT = """\
You are annotating a transcript of spontaneous Czech speech. A person was shown a drawing of a
lakeshore scene and asked to describe it aloud. The text is an automatic transcription.

Your job is to COUNT observable linguistic events and QUOTE the evidence for each count.
You are an annotator, not an evaluator.

RULES
- Quote evidence verbatim from the transcript, in Czech. Never translate or paraphrase.
- If a category has no instances, return an empty list. Empty is a valid answer.
- Never infer anything about the speaker: not their health, ability, intelligence, age,
  education or state of mind. Describe the language, never the person.
- Do not compare this speaker to anyone else or to any norm.
- Count occurrences, not impressions. Two hedges in one sentence are two entries.
- Return only JSON.

CATEGORIES
1.  named_entities - distinct objects, creatures or people explicitly named in the transcript.
    Lemmatise to the nominative singular and list each distinct entity exactly once. This is a
    SET of entities, not a list of mentions.
2.  specific_action_verbs - verbs naming a particular manner of action (jumps, flies, peeks out).
    List every occurrence.
3.  generic_verbs - verbs of bare existence, possession, location or unspecified movement.
    List every occurrence.
4.  complete_propositions - integer. Clauses with an explicit subject AND a predicate AND at
    least one further argument or adjunct. A bare noun phrase is not a proposition.
5.  locative_expressions - phrases placing something somewhere (on the shore, in the sky, on the
    left). List every occurrence.
6.  regions_referenced - which of "water", "land", "sky" are explicitly referred to. A SET;
    include a region only if something is actually located there in the speech.
7.  hedge_spans - expressions of uncertainty about what is depicted (maybe, probably, I don't
    know). List every occurrence.
8.  deictic_spans - places where the speaker points instead of naming (there, that, that thing).
    Include a span only when it substitutes for naming an entity.
9.  metacomment_spans - remarks about the speaker's own describing or remembering, or about the
    task. Not remarks about the picture.
10. repeated_content_lemmas - content words used more than once, with their counts.
11. self_corrections - integer. Restarts, replacements or retractions of something already begun.
12. diminutive_or_affective_forms - noun forms marked as diminutive or affectionate rather than
    neutral. List every occurrence in the form used.
13. quantity_expressions - numerals or quantifiers applied to things in the scene. List every
    occurrence.

Return exactly this JSON and nothing else:
{"named_entities": [], "specific_action_verbs": [], "generic_verbs": [],
 "complete_propositions": 0, "locative_expressions": [], "regions_referenced": [],
 "hedge_spans": [], "deictic_spans": [], "metacomment_spans": [],
 "repeated_content_lemmas": [{"lemma": "", "count": 0}], "self_corrections": 0,
 "diminutive_or_affective_forms": [], "quantity_expressions": []}
"""

LEAKY = ["two hidden", "two categories", "two classes", "hidden text categories", "diagnos",
         "impair", "dementia", "alzheim", "cognitive decline", "patient", "healthy",
         "control group", "clinical", "group of speakers"]
found = [w for w in LEAKY if w in V2_PROMPT.lower()]
assert not found, f"prompt leaks information about the task: {found}"
print(f"prompt is label-blind and domain-blind — OK ({len(V2_PROMPT)} chars)")

## 5. `StreamingLocalProvider`

Built on the library's `LocalProvider`, overriding the one method every call funnels through.
Four changes, each fixing something that broke an earlier run:

1. **Streaming.** A non-streamed 122B request on a long prompt can sit silent past the
   gateway's idle limit and come back as a 504. Streaming keeps tokens flowing, so the
   connection stays alive; we reassemble the chunks and parse at the end.
2. **`extra_body={"think": False}`.** Qwen 3.5 reasons before answering. The reasoning eats the
   token budget and the visible answer arrives empty — the `Invalid JSON response:` error with
   nothing after the colon. If the endpoint rejects the field we drop it and carry on.
3. **Retries with backoff** on timeouts, 5xx, empty completions and unparseable JSON — the
   stock provider retries only rate limits, so one bad response ends a 241-document run.
4. **`<think>` stripping**, for when the flag is ignored but the tags appear anyway.

The installed library is not modified.

In [ ]:
import openai
from openai import BadRequestError
from llm_feature_gen.providers.local_provider import LocalProvider

THINK_TAGS = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)


class StreamingLocalProvider(LocalProvider):
    """LocalProvider + streaming + no-think + retries. Same public interface."""

    def __init__(self, *args, stream=True, think=not DISABLE_THINKING,
                 request_timeout=REQUEST_TIMEOUT, **kwargs):
        super().__init__(*args, **kwargs)
        self.stream = stream
        self.extra_body = {} if think else {"think": False}
        self.request_timeout = request_timeout
        self.n_calls = 0
        self.errors = []

    def _complete(self, model, system_prompt, user_content, kwargs):
        """One request. Returns the assistant text."""
        if not self.stream:
            resp = self.client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system_prompt},
                          {"role": "user", "content": user_content}],
                temperature=self.temperature, max_tokens=self.max_tokens,
                timeout=self.request_timeout, **kwargs)
            return resp.choices[0].message.content or ""

        chunks = []
        stream = self.client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user", "content": user_content}],
            temperature=self.temperature, max_tokens=self.max_tokens,
            timeout=self.request_timeout, stream=True, **kwargs)
        for event in stream:
            if not event.choices:
                continue
            piece = getattr(event.choices[0].delta, "content", None)
            if piece:
                chunks.append(piece)
        return "".join(chunks)

    def _chat_json(self, deployment_name, system_prompt, user_content, json_mode=False):
        if json_mode and "JSON" not in system_prompt:
            system_prompt = system_prompt + " Respond in strict JSON format."

        kwargs = {}
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        if self.extra_body:
            kwargs["extra_body"] = dict(self.extra_body)

        backoff, last = 2, None
        for attempt in range(self.max_retries):
            try:
                self.n_calls += 1
                text = THINK_TAGS.sub("", self._complete(
                    deployment_name, system_prompt, user_content, kwargs)).strip()
                if not text:
                    raise ValueError("empty completion (model returned no visible content)")
                try:
                    parsed = json.loads(text)
                except Exception:
                    parsed = self._extract_json(text)
                if not isinstance(parsed, dict):
                    raise ValueError(f"unparseable response: {text[:160]}")
                return parsed

            except BadRequestError as e:
                msg = str(e)
                if json_mode and "json_object" in msg:
                    json_mode = False
                    kwargs.pop("response_format", None)
                    continue
                if "extra_body" in kwargs and ("think" in msg.lower() or "unknown" in msg.lower()):
                    kwargs.pop("extra_body")
                    print("  note: endpoint rejected {'think': False}; continuing without it")
                    continue
                last = e
            except Exception as e:
                last = e

            if attempt < self.max_retries - 1:
                print(f"    retry {attempt + 1}/{self.max_retries - 1} — "
                      f"{type(last).__name__}: {str(last)[:100]}")
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)

        self.errors.append(str(last)[:200])
        raise RuntimeError(f"gave up after {self.max_retries} attempts: {last}")


provider = None
if RUN_LLM:
    provider = StreamingLocalProvider(
        base_url=BASE_URL, api_key=API_KEY, default_text_model=MODEL,
        temperature=TEMPERATURE, max_tokens=MAX_TOKENS, max_retries=MAX_RETRIES)
    print(f"provider ready: {provider.base_url} | {provider.text_model} | "
          f"stream={provider.stream} | extra_body={provider.extra_body}")

### Smoke test

One tiny call before committing to 241. If this fails you will know in seconds.

In [ ]:
if RUN_LLM:
    t0 = time.time()
    probe = provider.text_features(
        ["Na břehu jsou kachny. Pes honí veverku."],
        prompt='Return only JSON: {"n_animals": <integer>}')[0]
    print(f"round trip {time.time() - t0:.1f}s | response: {probe}")
    assert probe, "empty response — check the endpoint, key and model name"

## 6. Run the extraction

One request per transcript. Each response is appended to `outputs/v2_raw_responses.jsonl` the
moment it arrives, and the loop skips documents already in that file. **If it dies on document
147, rerunning resumes at 147** — and a rerun after a completed run makes no calls at all.

In [ ]:
def load_raw(path=RAW_PATH):
    if not path.exists():
        return {}
    out = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            out[rec["file"]] = rec
    return out


def extract_all(df, provider, path=RAW_PATH):
    done = load_raw(path)
    todo = df[~df.file.isin(done)]
    print(f"{len(done)} already extracted, {len(todo)} to do")

    t0 = time.time()
    with path.open("a", encoding="utf-8") as fh:
        for i, row in enumerate(todo.itertuples(), 1):
            try:
                resp = provider.text_features([row.text], prompt=V2_PROMPT)[0]
                rec = {"file": row.file, "ok": True, "response": resp}
            except Exception as e:
                rec = {"file": row.file, "ok": False, "error": str(e)[:300], "response": {}}
                print(f"  FAILED {row.file}: {str(e)[:110]}")
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()                      # survive a kernel death
            done[row.file] = rec
            if i % 20 == 0 or i == len(todo):
                rate = (time.time() - t0) / i
                print(f"  {i}/{len(todo)}  ({rate:.1f}s/doc, "
                      f"~{rate * (len(todo) - i) / 60:.0f} min left)")
    return done


if RUN_LLM:
    t0 = time.time()
    raw = extract_all(train, provider)
    print(f"\nextraction finished in {(time.time() - t0) / 60:.1f} min "
          f"| {provider.n_calls} HTTP calls for {len(train)} documents")
else:
    raw = load_raw()
    print(f"loaded {len(raw)} cached responses (RUN_LLM is False)")

n_ok = sum(r["ok"] for r in raw.values())
print(f"{n_ok}/{len(raw)} succeeded, {len(raw) - n_ok} failed "
      f"({1 - n_ok / max(len(raw), 1):.1%} empty/failed)")

In [ ]:
# Record exactly what produced these numbers.
META_PATH.write_text(json.dumps({
    "model": MODEL, "base_url": BASE_URL, "prompt_version": PROMPT_VERSION,
    "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS, "streaming": True,
    "thinking_disabled": DISABLE_THINKING,
    "n_documents": len(train), "n_succeeded": n_ok,
    "prompt_sha256": __import__("hashlib").sha256(V2_PROMPT.encode()).hexdigest()[:16],
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "split": "train (241 labelled) — test set not read",
}, indent=2), encoding="utf-8")
print(META_PATH.read_text())

## 7. Is the extraction any good?

Before modelling, three checks. A representation built on unreliable extraction is not worth
comparing to anything.

In [ ]:
LIST_FIELDS = ["named_entities", "specific_action_verbs", "generic_verbs", "locative_expressions",
               "regions_referenced", "hedge_spans", "deictic_spans", "metacomment_spans",
               "diminutive_or_affective_forms", "quantity_expressions"]
INT_FIELDS = ["complete_propositions", "self_corrections"]
# fields whose entries are quoted from the transcript and can therefore be verified
SPAN_FIELDS = ["specific_action_verbs", "generic_verbs", "locative_expressions", "hedge_spans",
               "deictic_spans", "metacomment_spans", "diminutive_or_affective_forms",
               "quantity_expressions"]
VALID_REGIONS = {"water", "land", "sky"}


def structural_problems(obj):
    bad = [f for f in LIST_FIELDS if not isinstance(obj.get(f), list)]
    bad += [f for f in INT_FIELDS if not isinstance(obj.get(f), (int, float))]
    if set(obj.get("regions_referenced", [])) - VALID_REGIONS:
        bad.append("regions_referenced(bad values)")
    return bad


def groundedness(obj, transcript):
    """Share of quoted spans that really occur in the transcript. 1.0 = nothing invented."""
    spans = [s for f in SPAN_FIELDS for s in obj.get(f, []) if isinstance(s, str)]
    if not spans:
        return np.nan
    low = transcript.lower()
    return sum(s.lower().strip() in low for s in spans) / len(spans)


text_by_file = dict(zip(train.file, train.text))
checks = []
for f, rec in raw.items():
    if f not in text_by_file:
        continue
    obj = rec.get("response") or {}
    checks.append({"file": f, "ok": rec["ok"],
                   "n_problems": len(structural_problems(obj)) if rec["ok"] else np.nan,
                   "groundedness": groundedness(obj, text_by_file[f]) if rec["ok"] else np.nan})
checks = pd.DataFrame(checks)

print(f"empty / failed responses      : {(~checks.ok).mean():.1%}")
print(f"responses with a schema problem: {(checks.n_problems > 0).mean():.1%}")
print(f"mean evidence groundedness     : {checks.groundedness.mean():.3f} "
      f"(median {checks.groundedness.median():.3f})")
print(f"documents with any invented span: {(checks.groundedness < 1).mean():.1%}")

Groundedness is the honest reliability measure here, and it is one most LLM-feature papers
never report. A span the model quoted that is not in the transcript means the count that
produced it is fabricated. If this number is low, no modelling result below can be trusted.

## 8. Turn the responses into ML variables

Counts for every field, plus a **per-100-word rate** for each. The rates are the point: the
positive group produces shorter transcripts, so a raw count is partly a length measurement.
Reporting both is what separates a linguistic construct from a word counter.

The two set-valued fields become `named_entity_count` (distinct entities) and `regions_count`
(0–3), and we add one ratio, `specific_verb_ratio`, which is scale-free by construction.

In [ ]:
COUNT_FIELDS = {
    "named_entities_count":           "named_entities",
    "specific_action_verbs_count":    "specific_action_verbs",
    "generic_verbs_count":            "generic_verbs",
    "locative_expressions_count":     "locative_expressions",
    "hedge_count":                    "hedge_spans",
    "deictic_count":                  "deictic_spans",
    "metacomment_count":              "metacomment_spans",
    "diminutive_count":               "diminutive_or_affective_forms",
    "quantity_expressions_count":     "quantity_expressions",
}


def to_row(obj, transcript):
    n_word = max(len(transcript.split()), 1)
    row = {}
    for name, field in COUNT_FIELDS.items():
        v = obj.get(field)
        row[name] = len(v) if isinstance(v, list) else 0
    row["complete_propositions"] = int(obj.get("complete_propositions") or 0)
    row["self_corrections"] = int(obj.get("self_corrections") or 0)
    rcl = obj.get("repeated_content_lemmas")
    row["repeated_content_lemmas_count"] = (
        sum(int(d.get("count", 0)) for d in rcl if isinstance(d, dict)) if isinstance(rcl, list) else 0)
    regions = obj.get("regions_referenced")
    row["regions_count"] = len(set(regions) & VALID_REGIONS) if isinstance(regions, list) else 0

    counts = dict(row)
    row["n_word"] = n_word
    for k, v in counts.items():
        if k != "regions_count":
            row[f"{k}_per100"] = 100 * v / n_word
    sv, gv = row["specific_action_verbs_count"], row["generic_verbs_count"]
    row["specific_verb_ratio"] = sv / (sv + gv) if (sv + gv) else 0.0
    return row


usable = [f for f, r in raw.items() if r["ok"] and f in text_by_file]
V2 = pd.DataFrame([to_row(raw[f]["response"], text_by_file[f]) for f in usable], index=usable)
V2.index.name = "file"
V2 = V2.join(train.set_index("file")[["label"]])
V2.to_csv(FEAT_PATH)

print(f"{V2.shape[0]} documents x {V2.shape[1] - 1} features -> {FEAT_PATH}")
COUNTS = [c for c in V2.columns if not c.endswith("_per100")
          and c not in ("n_word", "label", "specific_verb_ratio")]
print("\ndistribution of the raw counts:")
print(V2[COUNTS].describe().T[["mean", "std", "min", "50%", "max"]].round(2).to_string())

### Do the counts just re-encode length?

r² against word count, for raw counts and for their rates. Raw counts should be
length-confounded; if the rates are too, the feature is measuring verbosity.

In [ ]:
r2 = pd.DataFrame([{
    "feature": c,
    "r2_raw": np.corrcoef(V2[c], V2.n_word)[0, 1] ** 2,
    "r2_rate": (np.corrcoef(V2[f"{c}_per100"], V2.n_word)[0, 1] ** 2
                if f"{c}_per100" in V2 else np.nan),
} for c in COUNTS if V2[c].std() > 0]).sort_values("r2_raw", ascending=False)
print(r2.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

### Which features separate the groups?

In [ ]:
from scipy.stats import mannwhitneyu, false_discovery_control
from sklearn.metrics import roc_auc_score

y = V2.label.astype(int).values
stats = []
for c in V2.columns.drop(["label", "n_word"]):
    if V2[c].std() == 0:
        continue
    a, b = V2.loc[y == 1, c], V2.loc[y == 0, c]
    auc = roc_auc_score(y, V2[c])
    stats.append({"feature": c, "median_neg": b.median(), "median_pos": a.median(),
                  "auc": max(auc, 1 - auc), "p": mannwhitneyu(a, b).pvalue})
stats = pd.DataFrame(stats)
stats["q_bh"] = false_discovery_control(stats.p.values, method="bh")
print(stats.sort_values("q_bh").head(15).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nfeatures with q < 0.05: {(stats.q_bh < 0.05).sum()} of {len(stats)}")

## 9. V1 versus V2

Every arm uses the **same** cross-validation: 10 repeats of stratified 5-fold on the same
documents, out-of-fold probabilities averaged across repeats, metrics computed once on that
average. Nothing here touches the test set.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sub = train[train.file.isin(V2.index)].set_index("file").loc[V2.index].reset_index()
y = V2.label.astype(int).values
logreg = lambda: LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)


def cv(name, model, X, n_repeats=10):
    probs = np.zeros((n_repeats, len(y)))
    for r in range(n_repeats):
        probs[r] = cross_val_predict(
            model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED + r),
            method="predict_proba")[:, 1]
    p = probs.mean(0); pred = (p >= .5).astype(int)
    row = {"representation": name, "auc": roc_auc_score(y, p),
           "balanced_accuracy": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro"),
           "accuracy": accuracy_score(y, pred)}
    print(f"{name:28} AUC {row['auc']:.3f}  BalAcc {row['balanced_accuracy']:.3f}  "
          f"macroF1 {row['macro_f1']:.3f}  Acc {row['accuracy']:.3f}")
    return row


results = [
    cv("Majority class", DummyClassifier(strategy="most_frequent"), V2[["n_word"]]),
    cv("Transcript length", make_pipeline(StandardScaler(), logreg()), V2[["n_word"]]),
]

# --- V1: the categorical features, if the earlier CSV is available ------------
V1_CSV = next((p for p in [Path("OutputsQwen/train_all_feature_values.csv"),
                           Path("outputs/features_v1_train.csv"),
                           Path("train_all_feature_values.csv")] if p.exists()), None)
if V1_CSV:
    v1 = pd.read_csv(V1_CSV)
    V1_COLS = [c for c in v1.columns if c not in ("File", "Class", "raw_llm_output")]
    v1 = v1.set_index("File").reindex(V2.index)
    ok_v1 = v1[V1_COLS].notna().all(axis=1)
    if ok_v1.sum() == len(V2):
        results.append(cv("V1 LLM features (categorical)",
                          make_pipeline(OneHotEncoder(handle_unknown="ignore", min_frequency=5),
                                        logreg()), v1[V1_COLS]))
    else:
        print(f"V1 features cover only {ok_v1.sum()}/{len(V2)} of these documents — skipped")
else:
    print("V1 feature CSV not found — skipping that arm")

# --- V2 ----------------------------------------------------------------------
RATES = [c for c in V2.columns if c.endswith("_per100")] + ["regions_count", "specific_verb_ratio"]
results += [
    cv("V2 raw counts", make_pipeline(StandardScaler(), logreg()), V2[COUNTS]),
    cv("V2 rates (length-normalised)", make_pipeline(StandardScaler(), logreg()), V2[RATES]),
    cv("V2 counts + rates", make_pipeline(StandardScaler(), logreg()),
       V2.drop(columns=["label"])),
    cv("TF-IDF char 3-5gram",
       make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                     sublinear_tf=True), logreg()), sub.text),
]

comparison = pd.DataFrame(results).sort_values("auc", ascending=False).reset_index(drop=True)
comparison.to_csv(OUT / "v2_comparison.csv", index=False)
print()
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## 10. Plots

In [ ]:
import matplotlib.pyplot as plt

INK, ACCENT, V2C, MUTED = "#14181C", "#125C63", "#9B4620", "#9AA39F"

# --- AUC comparison ----------------------------------------------------------
plot_df = comparison[comparison.representation != "Majority class"].sort_values("auc")
length_auc = float(comparison.loc[comparison.representation == "Transcript length", "auc"].iloc[0])

fig, ax = plt.subplots(figsize=(9, 0.55 * len(plot_df) + 1.6), dpi=120)
for i, r in enumerate(plot_df.itertuples()):
    is_v2 = r.representation.startswith("V2")
    ax.plot([length_auc, r.auc], [i, i], color="#E3E6E1", lw=6, solid_capstyle="butt", zorder=1)
    ax.scatter(r.auc, i, s=80, color=V2C if is_v2 else ACCENT, zorder=3,
               edgecolor="white", linewidth=1.5)
    ax.text(r.auc + 0.006, i, f"{r.auc:.3f}", va="center", fontsize=9,
            color=INK, family="monospace")
ax.axvline(length_auc, color=V2C, ls="--", lw=1, zorder=2)
ax.text(length_auc, len(plot_df) - 0.3, " word count", fontsize=8, color=V2C, va="bottom")
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df.representation, fontsize=10)
ax.set_xlabel(f"Area under ROC curve  (10×5-fold CV, {len(V2)} training documents)", fontsize=9.5)
ax.set_xlim(0.55, 1.0); ax.set_ylim(-0.6, len(plot_df) - 0.1)
ax.grid(axis="x", color="#E3E6E1", lw=0.8); ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#C2C7BE"); ax.tick_params(length=0)
ax.set_title(f"V2 LLM features vs. V1 — {MODEL}", fontsize=13, loc="left", pad=14, color=INK)
plt.tight_layout(); plt.savefig(OUT / "v2_auc_comparison.png", dpi=180, bbox_inches="tight",
                                facecolor="white")
plt.show()

In [ ]:
# --- feature distributions by class ------------------------------------------
show = (stats.sort_values("q_bh").feature.head(6).tolist()
        if len(stats) >= 6 else stats.feature.tolist())
fig, axes = plt.subplots(2, 3, figsize=(12, 6), dpi=120)
for ax, col in zip(axes.ravel(), show):
    for lab, colour, name in [(0, ACCENT, "control"), (1, V2C, "MCI / dementia")]:
        ax.hist(V2.loc[V2.label == lab, col], bins=12, alpha=.55, color=colour,
                label=name, density=True)
    ax.set_title(col, fontsize=10, color=INK)
    ax.grid(axis="y", color="#E3E6E1", lw=0.7); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(labelsize=8, length=0)
for ax in axes.ravel()[len(show):]:
    ax.axis("off")
axes.ravel()[0].legend(fontsize=8, frameon=False)
fig.suptitle("V2 feature distributions by class", fontsize=12, x=0.01, ha="left", color=INK)
plt.tight_layout(); plt.savefig(OUT / "v2_feature_distributions.png", dpi=180,
                                bbox_inches="tight", facecolor="white")
plt.show()

## 11. Conclusion

Read the table in section 9 against three reference points: the majority class (0.500), the
word count (~0.69) and the V1 categorical features (~0.69).

In [ ]:
def verdict(df):
    get = lambda n: float(df.loc[df.representation == n, "auc"].iloc[0]) if (df.representation == n).any() else None
    v1, length = get("V1 LLM features (categorical)"), get("Transcript length")
    v2 = max(a for n, a in zip(df.representation, df.auc) if n.startswith("V2"))
    print(f"best V2 arm        AUC {v2:.3f}")
    print(f"transcript length  AUC {length:.3f}")
    if v1: print(f"V1 categorical     AUC {v1:.3f}")
    print()
    if v1 and v2 > v1 + 0.05:
        print("V2 > V1. The redesign worked: asking for counts and evidence instead of ordinal")
        print("levels recovers signal the categorical encoding was discarding. The constructs")
        print("were right all along; the output format was the bottleneck.")
    elif v1 and abs(v2 - v1) <= 0.05:
        print("V2 = V1. The encoding was not the bottleneck. Before concluding anything about")
        print("the constructs, check section 7: if groundedness is below ~0.9 or the failure")
        print("rate is high, this is an extraction-reliability result, not a representation one.")
    elif v1:
        print("V2 < V1. Most likely the model is producing inconsistent counts rather than the")
        print("constructs being wrong. Section 7's groundedness and failure rates are where to")
        print("look, and repeating the extraction on 20 documents would confirm it.")
    if v2 <= length + 0.02:
        print("\nNote: V2 does not clearly beat counting the words in the transcript. Whatever")
        print("else is true, the representation is not yet measuring language rather than volume.")


verdict(comparison)

### What this notebook establishes for the consultation

* V1's weakness was diagnosed rather than guessed: coarse categorical levels, one degenerate
  feature, one length proxy, and a prompt that described the class structure to the model.
* V2 was designed from that diagnosis — observable counts, verbatim evidence, length
  normalisation done by us, and a prompt that is blind to both the labels and the domain.
* It was actually run, on all labelled training documents, with a large model, and the
  extraction was measured for reliability before anything was modelled.
* The comparison uses one protocol for every arm and never touches the test set.

**Reproducibility:** `outputs/v2_run_metadata.json` records the model, prompt version and
hash, temperature, token limit, document count and run date.
`outputs/v2_raw_responses.jsonl` holds every raw response, so the table above can be
regenerated with no further LLM calls, and an interrupted run resumes where it stopped.

**Not done here, deliberately:** the repeated-extraction stability study, rule mining,
augmentation and English transfer. Those are later-stage experiments and would only add noise
to this conversation.